In [1]:
"""
Complete BUS-BRA Training Script - Single File
All dependencies included, no external files needed
Modified for local PC with direct folder paths
"""

import os
import time
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import albumentations as A
import segmentation_models_pytorch as smp
from tqdm import tqdm


# =============================================================================
# CONFIGURATION - MODIFY THESE PATHS FOR YOUR PC
# =============================================================================

# Dataset paths - UPDATE THESE TO YOUR ACTUAL PATHS
BASE_DIR = "/srv/course/supernova/s202312054/project_file "
IMG_DIR = os.path.join(BASE_DIR, "image")  # Changed from "Images" to "image"
MASK_DIR = os.path.join(BASE_DIR, "musk")  # Changed from "Masks" to "musk"
CSV_PATH = os.path.join(BASE_DIR, "bus_data.csv")

# Output
CHECKPOINT_DIR = "/srv/course/supernova/s202312054/efficiencyNet/mobileNetV2/checkpoints"

# Model settings
ENCODER_NAME = 'mobilenet_v2'
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 16
NUM_EPOCHS = 500
LEARNING_RATE = 1e-4

# Data split
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
RANDOM_SEED = 42


# =============================================================================
# DATASET
# =============================================================================

class BUSBRADataset(Dataset):
    def __init__(self, images_path, masks_path, size=(256, 256), transform=None):
        self.images_path = images_path
        self.masks_path = masks_path
        self.size = size
        self.transform = transform
    
    def __len__(self):
        return len(self.images_path)
    
    def __getitem__(self, index):
        image = cv2.imread(self.images_path[index], cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(self.masks_path[index], cv2.IMREAD_GRAYSCALE)
        
        if self.transform is not None:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"]
        
        image = cv2.resize(image, self.size)
        mask = cv2.resize(mask, self.size, interpolation=cv2.INTER_NEAREST)
        
        image = image.astype(np.float32) / 255.0
        mask = (mask > 127).astype(np.float32)
        
        image = np.expand_dims(image, axis=0)
        mask = np.expand_dims(mask, axis=0)
        
        return torch.from_numpy(image), torch.from_numpy(mask)


def load_data(csv_path, img_dir, mask_dir):
    """
    Load and split data into train/val/test sets
    """
    print(f"\nChecking paths:")
    print(f"  CSV: {csv_path}")
    print(f"  Images: {img_dir}")
    print(f"  Masks: {mask_dir}")
    
    # Verify paths exist
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV file not found: {csv_path}")
    if not os.path.exists(img_dir):
        raise FileNotFoundError(f"Images directory not found: {img_dir}")
    if not os.path.exists(mask_dir):
        raise FileNotFoundError(f"Masks directory not found: {mask_dir}")
    
    df = pd.read_csv(csv_path)
    print(f"\nCSV loaded: {len(df)} entries")
    print(f"Columns: {list(df.columns)}")
    
    unique_cases = df["Case"].unique()
    print(f"Unique cases: {len(unique_cases)}")
    
    train_cases, temp_cases = train_test_split(
        unique_cases, test_size=(VAL_RATIO + TEST_RATIO), random_state=RANDOM_SEED
    )
    val_test_ratio = TEST_RATIO / (VAL_RATIO + TEST_RATIO)
    val_cases, test_cases = train_test_split(
        temp_cases, test_size=val_test_ratio, random_state=RANDOM_SEED
    )
    
    def get_paths(subset_df):
        img_paths, mask_paths = [], []
        missing_count = 0
        
        for _, row in subset_df.iterrows():
            img_id = row["ID"]
            
            # Try different extensions
            img_path = None
            for ext in ['.png', '.jpg', '.jpeg']:
                test_path = os.path.join(img_dir, img_id + ext)
                if os.path.exists(test_path):
                    img_path = test_path
                    break
            
            # For mask, handle the naming conversion
            mask_id = img_id.replace("bus_", "mask_")
            mask_path = None
            for ext in ['.png', '.jpg', '.jpeg']:
                test_path = os.path.join(mask_dir, mask_id + ext)
                if os.path.exists(test_path):
                    mask_path = test_path
                    break
            
            if img_path and mask_path:
                img_paths.append(img_path)
                mask_paths.append(mask_path)
            else:
                missing_count += 1
                if missing_count <= 3:  # Show first 3 missing files
                    print(f"  Missing: {img_id} (img: {img_path is not None}, mask: {mask_path is not None})")
        
        if missing_count > 0:
            print(f"  Total missing pairs: {missing_count}")
        
        return img_paths, mask_paths
    
    print("\nLoading train set...")
    train_x, train_y = get_paths(df[df["Case"].isin(train_cases)])
    
    print("\nLoading validation set...")
    val_x, val_y = get_paths(df[df["Case"].isin(val_cases)])
    
    print("\nLoading test set...")
    test_x, test_y = get_paths(df[df["Case"].isin(test_cases)])
    
    return (train_x, train_y), (val_x, val_y), (test_x, test_y)


# =============================================================================
# LOSS & METRICS
# =============================================================================

class DiceLoss(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, pred, target):
        pred = torch.sigmoid(pred)
        pred = pred.view(-1)
        target = target.view(-1)
        intersection = (pred * target).sum()
        dice = (2. * intersection + 1e-5) / (pred.sum() + target.sum() + 1e-5)
        return 1 - dice


class DiceBCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.dice = DiceLoss()
        self.bce = nn.BCEWithLogitsLoss()
    
    def forward(self, pred, target):
        return 0.5 * self.dice(pred, target) + 0.5 * self.bce(pred, target)


def dice_coefficient(pred, target):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    pred = pred.view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    dice = (2. * intersection + 1e-5) / (pred.sum() + target.sum() + 1e-5)
    return dice.item()


def iou_score(pred, target):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    pred = pred.view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    iou = (intersection + 1e-5) / (union + 1e-5)
    return iou.item()


# =============================================================================
# TRAINING LOOPS
# =============================================================================

def train_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    epoch_loss = 0.0
    epoch_dice = 0.0
    
    pbar = tqdm(loader, desc='Training')
    for images, masks in pbar:
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()
        
        dice = dice_coefficient(outputs, masks)
        epoch_loss += loss.item()
        epoch_dice += dice
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'dice': f'{dice:.4f}'})
    
    return epoch_loss / len(loader), epoch_dice / len(loader)


def validate_epoch(model, loader, loss_fn, device):
    model.eval()
    epoch_loss = 0.0
    epoch_dice = 0.0
    epoch_iou = 0.0
    
    pbar = tqdm(loader, desc='Validation')
    with torch.no_grad():
        for images, masks in pbar:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            loss = loss_fn(outputs, masks)
            
            dice = dice_coefficient(outputs, masks)
            iou = iou_score(outputs, masks)
            
            epoch_loss += loss.item()
            epoch_dice += dice
            epoch_iou += iou
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'dice': f'{dice:.4f}'})
    
    return epoch_loss / len(loader), epoch_dice / len(loader), epoch_iou / len(loader)


# =============================================================================
# MAIN
# =============================================================================

def main():
    print("=" * 60)
    print("BUS-BRA TRAINING - 500 EPOCHS")
    print("Modified for Local PC")
    print("=" * 60)
    
    # Set seeds
    np.random.seed(RANDOM_SEED)
    torch.manual_seed(RANDOM_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(RANDOM_SEED)
    
    # Create output directory
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f"\nCheckpoints will be saved to: {os.path.abspath(CHECKPOINT_DIR)}")
    
    # Device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\nDevice: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("WARNING: Running on CPU. Training will be slow!")
    
    # Load data
    print("\n" + "=" * 60)
    print("LOADING DATA")
    print("=" * 60)
    
    try:
        (train_x, train_y), (val_x, val_y), (test_x, test_y) = load_data(CSV_PATH, IMG_DIR, MASK_DIR)
    except Exception as e:
        print(f"\nERROR loading data: {e}")
        print("\nPlease check:")
        print("1. BASE_DIR path is correct")
        print("2. Folders 'image' and 'musk' exist")
        print("3. File 'bus_data.csv' exists")
        return
    
    print(f"\n✓ Data loaded successfully!")
    print(f"  Train: {len(train_x)} samples")
    print(f"  Val: {len(val_x)} samples")
    print(f"  Test: {len(test_x)} samples")
    
    if len(train_x) == 0:
        print("\nERROR: No training samples found!")
        print("Please check your image and mask file naming conventions")
        return
    
    # Augmentation
    train_transform = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.RandomRotate90(p=0.3),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,
                           rotate_limit=15, p=0.5),
        A.ElasticTransform(alpha=1, sigma=50, p=0.2),
        A.RandomBrightnessContrast(brightness_limit=0.2,
                                   contrast_limit=0.2, p=0.4),
        A.GaussNoise(p=0.2),
    ])
    
    # Datasets
    train_dataset = BUSBRADataset(train_x, train_y, IMAGE_SIZE, train_transform)
    val_dataset = BUSBRADataset(val_x, val_y, IMAGE_SIZE, None)
    
    # DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    
    # Model
    print("\n" + "=" * 60)
    print("BUILDING MODEL")
    print("=" * 60)
    
    model = smp.Unet(
        encoder_name=ENCODER_NAME,
        encoder_weights='imagenet',
        in_channels=1,
        classes=1,
        activation=None
    ).to(device)
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\nModel: U-Net with {ENCODER_NAME} encoder")
    print(f"Total parameters: {total_params:,}")
    print(f"Image size: {IMAGE_SIZE}")
    print(f"Batch size: {BATCH_SIZE}")
    
    # Loss & Optimizer
    loss_fn = DiceBCELoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    
    # Track best model
    best_val_dice = 0.0
    best_epoch = 0
    
    # Training history
    history = []
    
    # Training
    print("\n" + "=" * 60)
    print("STARTING TRAINING")
    print("=" * 60)
    print(f"Total epochs: {NUM_EPOCHS}")
    print(f"Learning rate: {LEARNING_RATE}")
    print("=" * 60)
    
    for epoch in range(1, NUM_EPOCHS + 1):
        start_time = time.time()
        
        train_loss, train_dice = train_epoch(model, train_loader, optimizer, loss_fn, device)
        val_loss, val_dice, val_iou = validate_epoch(model, val_loader, loss_fn, device)
        
        elapsed = time.time() - start_time
        mins = int(elapsed / 60)
        secs = int(elapsed - (mins * 60))
        
        print(f"\nEpoch [{epoch:03d}/{NUM_EPOCHS}] {mins}m {secs}s")
        print(f"  Train Loss: {train_loss:.4f} | Train Dice: {train_dice:.4f}")
        print(f"  Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f} | Val IoU: {val_iou:.4f}")
        
        scheduler.step(val_loss)
        
        # Track best
        if val_dice > best_val_dice:
            best_val_dice = val_dice
            best_epoch = epoch
            torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, 'best_model.pth'))
            print(f"  ★ New best Val Dice: {val_dice:.4f} - Model saved!")
        
        # Save history
        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'train_dice': train_dice,
            'val_loss': val_loss,
            'val_dice': val_dice,
            'val_iou': val_iou
        })
        
        # Save model every 50 epochs
        if epoch % 50 == 0:
            torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, f'model_epoch_{epoch}.pth'))
            print(f"  Checkpoint saved: model_epoch_{epoch}.pth")
    
    # Save final model
    torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, 'final_model.pth'))
    
    # Training complete
    print("\n" + "=" * 60)
    print("TRAINING COMPLETE!")
    print("=" * 60)
    print(f"\nBest Performance:")
    print(f"  Val Dice: {best_val_dice:.4f}")
    print(f"  Epoch: {best_epoch}")
    print(f"\nSaved Models:")
    print(f"  Best model: {os.path.join(CHECKPOINT_DIR, 'best_model.pth')}")
    print(f"  Final model: {os.path.join(CHECKPOINT_DIR, 'final_model.pth')}")
    
    # Save training history
    import json
    history_file = os.path.join(CHECKPOINT_DIR, 'training_history.json')
    with open(history_file, 'w') as f:
        json.dump(history, f, indent=2)
    print(f"  Training history: {history_file}")
    
    # Save summary
    summary = {
        'total_epochs': NUM_EPOCHS,
        'best_val_dice': best_val_dice,
        'best_epoch': best_epoch,
        'final_val_dice': history[-1]['val_dice'],
        'train_samples': len(train_x),
        'val_samples': len(val_x),
        'test_samples': len(test_x),
        'image_size': IMAGE_SIZE,
        'batch_size': BATCH_SIZE,
        'encoder': ENCODER_NAME
    }
    
    summary_file = os.path.join(CHECKPOINT_DIR, 'training_summary.json')
    with open(summary_file, 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"  Training summary: {summary_file}")
    
    print("\n" + "=" * 60)


if __name__ == "__main__":
    main()

/opt/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BUS-BRA TRAINING - 500 EPOCHS
Modified for Local PC


PermissionError: [Errno 13] Permission denied: '/srv/course'